In [1]:
!pip install -q git+https://github.com/RayenSansa03/aeroguard.git@main --force-reinstall --no-deps

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from aeroguard.donnees_ml import preparer_xy, poids_classes

DOSSIER = "/content/drive/MyDrive/aeroguard"   # ← le même que dans le notebook 22
vols = pd.read_parquet(f"{DOSSIER}/data/processed/vols_ncmapss.parquet")

X_train, y_train = preparer_xy(vols, "train")
X_val, y_val = preparer_xy(vols, "val")

print("Répartition train :")
print(y_train.value_counts(normalize=True).round(3))
print("\nPoids balanced :", {k: round(v, 3) for k, v in poids_classes(y_train).items()})

Mounted at /content/drive
Répartition train :
hs
1    0.705
0    0.295
Name: proportion, dtype: float64

Poids balanced : {0: 1.695, 1: 0.709}


In [2]:
from sklearn.dummy import DummyClassifier
from aeroguard.evaluation import taux_par_classe

bete = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(taux_par_classe(y_val, bete.predict(X_val)))

{'exactitude': 0.702075702075702, 'uses_detectes': 1.0, 'sains_reconnus': 0.0}


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from aeroguard.evaluation import metriques

resultats = {}
for nom, poids in [("logistique", None), ("logistique_balanced", "balanced")]:
    modele = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight=poids),
    )
    modele.fit(X_train, y_train)
    pred = modele.predict(X_val)
    proba = modele.predict_proba(X_val)[:, 1]
    resultats[nom] = {**metriques(y_val, pred, proba), **taux_par_classe(y_val, pred)}

tableau = pd.DataFrame(resultats).T
colonnes = ["sains_reconnus", "rappel", "precision", "fp", "fn", "f1", "f1_macro", "pr_auc"]
tableau[colonnes].round(3)

,sains_reconnus,rappel,precision,fp,fn,f1,f1_macro,pr_auc
logistique,0.840,0.932,0.932,39.0,39.0,0.932,0.886,0.987
logistique_balanced,0.947,0.873,0.975,13.0,73.0,0.921,0.882,0.987


In [4]:
from aeroguard.evaluation import ajouter_au_leaderboard

s = resultats["logistique_balanced"]
cles = ["precision", "rappel", "f1", "f1_macro", "pr_auc"]
ajouter_au_leaderboard(
    f"{DOSSIER}/results/leaderboard.csv",
    "logistique_balanced",
    {k: s[k] for k in cles},
)
pd.read_csv(f"{DOSSIER}/results/leaderboard.csv").sort_values("f1_macro", ascending=False)

,modele,f1_macro,pr_auc,rappel,precision,f1,fp,fn
2,logistique,0.886000,0.987000,0.932000,0.932000,0.932000,39.0,39.0
3,logistique_balanced,0.882083,0.986936,0.873043,0.974757,0.921101,NaN,NaN
1,copieur_knn,0.763000,0.903000,0.843000,0.866000,0.855000,75.0,90.0
0,bete,0.412000,0.702000,1.000000,0.702000,0.825000,244.0,0.0
